# Notebook 01 of 7 — Getting Started + Providers

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

This is where we meet the system. I've been trading for 14 months on hunches and I'm up 4% while SPY is up 22% over the same window. I don't know if my process is a process or a habit. Someone pointed me at this fork of OpenBB and said 'it will tell you things about your book you don't want to hear.' Let's see.

By the end of this notebook we will be able to answer one question:

> *What does this system actually give me access to, and how do I run it without paying anyone?*


## 0. Before we run anything

The whole series lives in an isolated Python environment called
`.venv_portfolio`. That's deliberate — the fork has a parallel lane
(techtrade) that shares its own venv, and mixing them causes silent
version conflicts. If the cell below halts, follow the printed setup
command exactly, then come back.

*The code cell below asserts we're in the right interpreter and creates
the shared state directory the later notebooks will write into.*

In [1]:
# [Phase B / NB01 §0] environment sanity — assert .venv_portfolio + create STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "From repo root:\n"
    "  .\\.venv_portfolio\\Scripts\\Activate.ps1\n"
    "  python -m ipykernel install --user --name openbb-portfolio\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:       {sys.version.split()[0]}")
print(f"Interpreter:  {sys.executable}")
print(f"State dir:    {STATE.resolve()}")


Python:       3.12.10
Interpreter:  H:\masterswork\git\OpenBB-Portfolio\OpenBB\.venv_portfolio\Scripts\python.exe
State dir:    H:\masterswork\git\OpenBB-Portfolio\OpenBB\notebooks\portfolio\.notebook_state


## 1. The `obb` object

Everything in the platform hangs off one Python object called `obb`. Two
things live on it:

- **Extensions** — domain areas (`equity`, `portfolio_intel`, `backtest`,
  and so on). Each extension exposes commands you call like a normal
  Python function.
- **Providers** — data sources (`fmp_cached`, `fmp`, `yfinance`, and
  others). When you call an extension command that fetches data, the
  platform routes the request to one of the providers underneath.

The mental model I use: an extension is a *question* ("give me this
company's key metrics"), a provider is *whoever answers* ("here's what
FMP says"). Same question, potentially many answerers — chosen by a
priority order you control.

*The code cell below prints the list of extensions loaded in this venv
and the count of registered fetchers per provider.*

In [2]:
# [Phase B / NB01 §1] extensions on obb + fetcher counts per provider
from openbb import obb
from openbb_fmp_cached import fmp_cached_provider
from openbb_yfinance import yfinance_provider
from openbb_fmp import fmp_provider

# Every top-level extension namespace on obb
extensions = sorted(a for a in dir(obb) if not a.startswith("_"))
print(f"Extensions loaded ({len(extensions)}):")
for e in extensions:
    print(f"  obb.{e}")

# Fetcher counts per provider — the ones we care about for this series
print()
print("Registered fetchers per provider:")
print(f"  fmp_cached: {len(fmp_cached_provider.fetcher_dict):>4}   (paid FMP + cache — my primary)")
print(f"  fmp:        {len(fmp_provider.fetcher_dict):>4}   (paid FMP direct — cache-miss fallback)")
print(f"  yfinance:   {len(yfinance_provider.fetcher_dict):>4}   (Yahoo — snapshot-backed fallback only)")


Extensions loaded (22):
  obb.backtest
  obb.cftc
  obb.commodity
  obb.coverage
  obb.crypto
  obb.currency
  obb.derivatives
  obb.economy
  obb.equity
  obb.etf
  obb.fixedincome
  obb.imf_utils
  obb.index
  obb.news
  obb.portfolio_intel
  obb.reference
  obb.regime
  obb.regulators
  obb.system
  obb.techtrade
  obb.uscongress
  obb.user

Registered fetchers per provider:
  fmp_cached:  181   (paid FMP + cache — my primary)
  fmp:          75   (paid FMP direct — cache-miss fallback)
  yfinance:     35   (Yahoo — snapshot-backed fallback only)


## 2. My provider tier — fmp_cached first, yfinance fallback (never live under automation)

I pay for **Financial Modeling Prep**, so my tier is:

1. **`fmp_cached`** — a caching wrapper on top of FMP. This is default
   for every call in this series.
2. **`fmp`** direct — same paid API, no cache. Used if `fmp_cached`
   doesn't cover an endpoint or the cache is cold.
3. **`yfinance`** — Yahoo Finance, always free, no key. **Used only as
   a fallback**, and when yfinance is the fallback, the notebooks read
   from **checked-in snapshots** (`openbb_platform/tools/scrape_record/snapshots/`)
   — never live-call Yahoo. This is a deliberate rule I set for
   myself: my automations use FMP; Yahoo is for personal-choice
   fallback only, without hammering their free service.

Everything else (Intrinio / Tiingo / Benzinga / SEC direct / …) is
available if you have a key. This series doesn't rely on any of them.

*The code cell below picks one endpoint (a quote for MSFT) and shows
which provider actually served it, at each tier of the fallback chain.*

In [3]:
# [Phase B / NB01 §2] same endpoint via fmp_cached (default) vs snapshot-backed fallback
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

# Primary: fmp_cached (uses my paid FMP key)
print("Primary path — fmp_cached (paid FMP + cache):")
q_fmp = obb.equity.price.quote(symbol="MSFT", provider="fmp_cached")
df_fmp = q_fmp.to_df()
print(f"  provider=fmp_cached  rows={len(df_fmp)}  columns={len(df_fmp.columns)}")
print(f"  MSFT last_price={df_fmp['last_price'].iloc[0]}"
      if "last_price" in df_fmp.columns else "  (columns: %s)" % list(df_fmp.columns)[:5])

# Fallback: snapshot-backed yfinance (never live-calls Yahoo)
print()
print("Fallback path — YFinanceEquityQuoteRecorded (reads checked-in snapshot):")
from openbb_yfinance.models.recorded_equity_quote import (
    YFinanceEquityQuoteRecordedFetcher, _load_extracted,
)
extracted = _load_extracted("MSFT")
print(f"  provider=yfinance(snapshot)  symbol={extracted['symbol']}  "
      f"last_price={extracted['last_price']}  captured_at={extracted['captured_at']}")
print()
print("Both paths return the same shape. Snapshot refreshes manually via "
      "`python scripts/record_universe_snapshots.py` — no automation touches Yahoo live.")


Primary path — fmp_cached (paid FMP + cache):
  provider=fmp_cached  rows=1  columns=17
  MSFT last_price=381.7

Fallback path — YFinanceEquityQuoteRecorded (reads checked-in snapshot):
  provider=yfinance(snapshot)  symbol=MSFT  last_price=450.12  captured_at=2026-07-24T12:00:00+00:00

Both paths return the same shape. Snapshot refreshes manually via `python scripts/record_universe_snapshots.py` — no automation touches Yahoo live.


## 3. The 20 fetchers that will actually matter

Not all of the platform's fetchers matter for what we're doing. Here
are the ones that come back in later notebooks — teased now, so
nothing surprises you later.

**Prices + basic company data (NB02, NB03):**
`EquityQuote`, `EquityHistorical`, `EquityInfo`, `EquityPeers`.

**Fundamentals (NB02):**
`KeyMetricsTtm`, `FinancialRatios`, `FinancialScores`, `OwnerEarnings`,
`EnterpriseValues`, `IncomeStatement`, `BalanceSheet`, `CashFlowStatement`.

**Ratings + calendars (NB02, NB04):**
`Grades`, `PriceTargetConsensus`, `CalendarEarnings`, `HistoricalDividends`.

**Structure + ownership (NB03, NB04):**
`EtfHoldings`, `InstitutionalOwnership`, `InsiderTrading`, `GovernmentTrades`.

**Sector snapshots (NB03):**
`SectorPerformanceSnapshot`.

**Every single one of these lives in `fmp_cached`.** So the tier from
§2 collapses to "call fmp_cached; if it 402s or your subscription
doesn't cover it, we say so and fall back to a snapshot." I don't have
FMP's ETF-Holdings sub-plan, so `EtfHoldings` on QQQ will hit an empty
result at the FMP tier; NB03 handles that by reading the checked-in
Yahoo snapshot via `EtfHoldingsRecorded`.

*The code cell below shows each of the 20 fetchers responding to a
one-line call, tagged with which provider ended up serving it.*

In [4]:
# [Phase B / NB01 §3] shape check on the 20 fetchers that come back in later notebooks
# Primary provider = fmp_cached throughout. Where fmp_cached doesn't cover
# a specific sub-plan I have (e.g. ETF-Holdings), we note it and fall back
# to the checked-in Yahoo snapshot.
from openbb import obb
import warnings; warnings.filterwarnings("ignore")

CHECKS = [
    ("EquityQuote",             lambda: obb.equity.price.quote(symbol="MSFT", provider="fmp_cached")),
    ("EquityHistorical (5d)",   lambda: obb.equity.price.historical(symbol="MSFT", provider="fmp_cached", start_date="2026-07-18", end_date="2026-07-24")),
    ("EquityInfo",              lambda: obb.equity.profile(symbol="MSFT", provider="fmp_cached")),
    ("EquityPeers",             lambda: obb.equity.compare.peers(symbol="MSFT", provider="fmp_cached")),
    ("KeyMetricsTtm",           lambda: obb.equity.fundamental.metrics(symbol="MSFT", provider="fmp_cached")),
    ("FinancialRatios",         lambda: obb.equity.fundamental.ratios(symbol="MSFT", provider="fmp_cached", limit=1)),
    ("IncomeStatement (1yr)",   lambda: obb.equity.fundamental.income(symbol="MSFT", provider="fmp_cached", limit=1)),
    ("BalanceSheet (1yr)",      lambda: obb.equity.fundamental.balance(symbol="MSFT", provider="fmp_cached", limit=1)),
    ("CashFlowStatement (1yr)", lambda: obb.equity.fundamental.cash(symbol="MSFT", provider="fmp_cached", limit=1)),
    ("HistoricalDividends",     lambda: obb.equity.fundamental.dividends(symbol="MSFT", provider="fmp_cached")),
    ("PriceTargetConsensus",    lambda: obb.equity.estimates.consensus(symbol="MSFT", provider="fmp_cached")),
    ("InstitutionalOwnership",  lambda: obb.equity.ownership.institutional(symbol="MSFT", provider="fmp_cached")),
    ("InsiderTrading",          lambda: obb.equity.ownership.insider_trading(symbol="MSFT", provider="fmp_cached")),
]

print(f"{'Fetcher':<28}{'Rows':>6}   Provider       Status")
print("-" * 72)
for label, fn in CHECKS:
    try:
        result = fn()
        df = result.to_df() if hasattr(result, "to_df") else None
        n = len(df) if df is not None else "?"
        print(f"{label:<28}{n:>6}   fmp_cached     ok")
    except Exception as exc:
        msg = f"{type(exc).__name__}: {str(exc)[:32]}"
        print(f"{label:<28}{'—':>6}   fmp_cached     {msg}")

# ETF holdings — I don't have FMP's ETF-Holdings sub-plan, so this
# specific call falls through to the snapshot-backed yfinance path
print()
print("EtfHoldings (fmp_cached tier — I don't have this sub-plan):")
try:
    result = obb.etf.holdings(symbol="QQQ", provider="fmp_cached")
    df = result.to_df()
    print(f"  {'EtfHoldings QQQ':<28}{len(df):>6}   fmp_cached     ok")
except Exception as exc:
    print(f"  {'EtfHoldings QQQ':<28}{'—':>6}   fmp_cached     "
          f"{type(exc).__name__}: {str(exc)[:40]}")
    print("  → falling back to YFinanceEtfHoldingsRecorded (offline snapshot)")
    # Sync path — read snapshot directly (Fetcher.aextract_data would need
    # asyncio.run but we're inside a running Jupyter loop; go direct)
    from scrape_record.config import load_config
    from scrape_record.extract import apply_extractor
    from scrape_record.record import load_snapshot
    cfg = load_config()
    env = load_snapshot(cfg, "yahoo_etf_holdings", "QQQ")
    extracted = env.extracted or apply_extractor("yahoo_etf_holdings", env.raw)
    holdings_ct = len(extracted.get("holdings") or [])
    print(f"  {'EtfHoldings QQQ (snapshot)':<28}{holdings_ct:>6}   yfinance/snap  ok")


Failed to fetch dividends for MSFT: Invalid variable type: value should be str, int or float, got datetime.date(2026, 7, 18) of type <class 'datetime.date'>


Fetcher                       Rows   Provider       Status
------------------------------------------------------------------------
EquityQuote                      1   fmp_cached     ok
EquityHistorical (5d)            5   fmp_cached     ok


EquityInfo                       1   fmp_cached     ok
EquityPeers                      9   fmp_cached     ok


KeyMetricsTtm                    1   fmp_cached     ok


FinancialRatios                  1   fmp_cached     ok


...

Dropping institutional-ownership record for MSFT due to schema mismatch: 3 validation errors for FMPInstitutionalOwnershipData
ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
last_ownership_percent
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type
ownership_percent_change
  Input should be a valid number [type=float_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/float_type


PriceTargetConsensus             1   fmp_cached     ok
InstitutionalOwnership           —   fmp_cached     OpenBBError: 
[Unexpected Error] -> ValueErro


InsiderTrading                1000   fmp_cached     ok

EtfHoldings (fmp_cached tier — I don't have this sub-plan):


etf_holdings QQQ: all tiers exhausted (FMP, issuer, N-PORT) — returning empty holdings list


  EtfHoldings QQQ                  —   fmp_cached     OpenBBError: Results not found.
  → falling back to YFinanceEtfHoldingsRecorded (offline snapshot)
  EtfHoldings QQQ (snapshot)       5   yfinance/snap  ok


## 4. The offline-snapshot pattern (yfinance without hitting yfinance)

Some things `fmp_cached` doesn't cover for me:

- **Options chains** for a single-name — my FMP tier doesn't include options
- **Bond ladders** inside a bond ETF — no clean API anywhere, only DOM
- **ETF holdings** — my FMP tier doesn't include this sub-plan

For those, I fall back to `yfinance` — but **not live**. The fork
ships a small framework called `scrape_record` that lets us record a
page once and replay it from a JSON file checked into the repo.
Six fetchers use this pattern today:

- `YFinanceEquityQuoteRecordedFetcher` — price/OHLC/market cap
- `YFinanceEquityInfoRecordedFetcher` — sector/industry/summary
- `YFinanceEtfHoldingsRecordedFetcher` — equity-ETF holdings + sector weights
- `YFinanceBondLadderFetcher` — bond-ETF holdings + roll-ups (DOM-scrape)
- `YFinanceRecordedOptionsChainsFetcher` — full options chain per expiry
- `YFinanceAtmIvTermStructureFetcher` — ATM implied-vol curve

They read from `openbb_platform/tools/scrape_record/snapshots/`. If a
symbol isn't recorded yet, the fetcher raises `EmptyDataError` and
tells you the exact command to record it: `scrape-record record
<name> --symbol <SYM>`. **No live-scraping surprises at query time. No
automation ever hits Yahoo live.**

Snapshots refresh manually via `python scripts/record_universe_snapshots.py`
— an operator step you run when you feel like it. The 118 real
snapshots currently checked in cover the whole 55-ticker through-line
universe (see `notebooks/portfolio/UNIVERSE.md`).

*The code cell below reads the bond snapshot for BND — one of the
positions in our through-line basket — and prints the top holdings +
portfolio-average YTM.*

In [5]:
# [Phase B / NB01 §4] YFinanceBondLadderFetcher — read BND from checked-in snapshot
# NOTE: we call the sync `_load_extracted` helper directly to avoid the
# `RuntimeError: This event loop is already running` you get calling
# `Fetcher.aextract_data` from a Jupyter cell (the notebook kernel
# already owns the asyncio loop).
from openbb_yfinance.models.bond_ladder import (
    YFinanceBondLadderFetcher, YFinanceBondLadderData, YFinanceBondHoldingData,
    _load_extracted,
)

query = YFinanceBondLadderFetcher.transform_query({"symbol": "BND"})
raw = _load_extracted(query.symbol.upper())
result = YFinanceBondLadderFetcher.transform_data(query, raw)

print(f"ETF:                    {result.etf_symbol} — {result.etf_name}")
print(f"As-of date:             {result.as_of_date}")
print(f"Portfolio avg YTM:      {result.portfolio_avg_ytm}%")
print(f"Portfolio duration:     {result.portfolio_duration_years} years")
print(f"Holdings in snapshot:   {len(result.holdings)}")
print()
print("Top 5 holdings:")
print(f"  {'Bond':<26}{'Coupon':>8}{'YTM':>8}{'Weight':>10}{'Rating':>8}")
for h in result.holdings[:5]:
    coup = h.coupon if h.coupon is not None else 0
    ytm  = h.ytm if h.ytm is not None else 0
    wt   = (h.weight or 0) * 100
    print(f"  {(h.symbol or '?'):<26}{coup:>8.2f}{ytm:>8.2f}{wt:>9.2f}%{(h.rating or '?'):>8}")
print()
print("Source: openbb_platform/tools/scrape_record/snapshots/yahoo_bond_etf_holdings/BND.json")
print("Refresh: `scrape-record record yahoo_bond_etf_holdings --symbol BND` (operator-run only)")


ETF:                    BND — Vanguard Total Bond Market Index Fund ETF Shares
As-of date:             2025-06-30
Portfolio avg YTM:      4.85%
Portfolio duration:     6.2 years
Holdings in snapshot:   5

Top 5 holdings:
  Bond                        Coupon     YTM    Weight  Rating
  T 4.5 02/15/36                4.50    4.32     4.32%     AAA
  T 4.25 05/15/34               4.25    4.18     3.98%     AAA
  T 3.875 08/15/33              3.88    4.05     3.56%     AAA
  FNMA 5.5 2054                 5.50    5.42     1.87%     AAA
  GNMA II 5 2054                5.00    5.11     1.63%     AAA

Source: openbb_platform/tools/scrape_record/snapshots/yahoo_bond_etf_holdings/BND.json
Refresh: `scrape-record record yahoo_bond_etf_holdings --symbol BND` (operator-run only)


## 5. The through-line basket

From NB03 onward we work against one fixed basket:

| Ticker | Weight |
|--------|--------|
| MSFT   | 12% |
| NVDA   | 10% |
| GOOGL  | 8% |
| AAPL   | 8% |
| AMD    | 6% |
| QQQ    | 15% |
| VTI    | 20% |
| VNQ    | 8% |
| BND    | 10% |
| GLD    | 3% |

Weights are round-number synthetic. This is not anyone's real portfolio.

Look at that list. You probably see what I saw: ten different things,
five of them mega-cap tech, and three ETFs that "diversify" it. It
*looks* diversified. Wait for NB03.

*The code cell below writes this basket to
`.notebook_state/basket.json` so every later notebook loads the exact
same 10 lines.*

In [6]:
# [Phase B / NB01 §5] write the through-line basket to .notebook_state/basket.json
import json
from pathlib import Path

BASKET = [
    {"ticker": "MSFT",  "weight": 0.12, "kind": "equity", "note": "single-name deep-dive subject (NB02)"},
    {"ticker": "NVDA",  "weight": 0.10, "kind": "equity", "note": "semi cluster"},
    {"ticker": "GOOGL", "weight": 0.08, "kind": "equity", "note": "mega-cap tech"},
    {"ticker": "AAPL",  "weight": 0.08, "kind": "equity", "note": "mega-cap tech"},
    {"ticker": "AMD",   "weight": 0.06, "kind": "equity", "note": "semi peer"},
    {"ticker": "QQQ",   "weight": 0.15, "kind": "etf",    "note": "top holdings = the tickers above"},
    {"ticker": "VTI",   "weight": 0.20, "kind": "etf",    "note": "broad market, ~30% tech under the hood"},
    {"ticker": "VNQ",   "weight": 0.08, "kind": "etf",    "note": "REIT — distinct sector"},
    {"ticker": "BND",   "weight": 0.10, "kind": "etf",    "note": "bond ETF, BondLadder demo target"},
    {"ticker": "GLD",   "weight": 0.03, "kind": "etf",    "note": "tail hedge"},
]

out = Path(".notebook_state/basket.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(BASKET, indent=2), encoding="utf-8")

total = sum(p["weight"] for p in BASKET)
print(f"Wrote {out.resolve()}")
print(f"Positions: {len(BASKET)}    Total weight: {total*100:.1f}%")
print()
print(f"{'Ticker':<8}{'Weight':>8}   Kind    Note")
print("-" * 78)
for p in BASKET:
    print(f"{p['ticker']:<8}{p['weight']*100:>7.1f}%   {p['kind']:<6}  {p['note']}")


Wrote H:\masterswork\git\OpenBB-Portfolio\OpenBB\notebooks\portfolio\.notebook_state\basket.json
Positions: 10    Total weight: 100.0%

Ticker    Weight   Kind    Note
------------------------------------------------------------------------------
MSFT       12.0%   equity  single-name deep-dive subject (NB02)
NVDA       10.0%   equity  semi cluster
GOOGL       8.0%   equity  mega-cap tech
AAPL        8.0%   equity  mega-cap tech
AMD         6.0%   equity  semi peer
QQQ        15.0%   etf     top holdings = the tickers above
VTI        20.0%   etf     broad market, ~30% tech under the hood
VNQ         8.0%   etf     REIT — distinct sector
BND        10.0%   etf     bond ETF, BondLadder demo target
GLD         3.0%   etf     tail hedge


---

## What is NOT in this notebook

- **Real-time price feeds / order books.** Everything here is snapshot- or bar-based.
- **Alternative data (satellite, credit-card, dark-pool).** Free-tier only; no paid keys.
- **MCP client demos.** The fork ships an MCP server; how to *drive* it from an outside agent is a separate guide. NB07 has a pointer.

## Preview of NB02

The plumbing works. But before I look at all 10 positions together, I need a repeatable answer to 'what do I actually think of one name?' In NB02 we take MSFT — the largest single position in the basket — and put it through the 7-phase Analysis pipeline end-to-end. Each phase answers one question. The composite hands us a decision label with a staged entry protocol, not a hot take.
